In [1]:
import os

print("=== Dataset contents ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

=== Dataset contents ===
/kaggle/input/datasets/aniruddhabhagat/hinglishdb-finetuning-data/hinglish_finetuning_ready/dev.json
/kaggle/input/datasets/aniruddhabhagat/hinglishdb-finetuning-data/hinglish_finetuning_ready/train.json
/kaggle/input/datasets/aniruddhabhagat/hinglishdb-finetuning-data/hinglish_finetuning_ready/test.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/ex1_createtable_predictions.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/finetuning_ready/dev.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/finetuning_ready/train.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/finetuning_ready/test.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/splits/train_en.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/splits/test_en.json
/kaggle/input/datasets/aniruddhabhagat/hi

In [2]:
!pip install "transformers==4.43.4" -q
!pip install "peft==0.11.1" -q
!pip install "bitsandbytes==0.49.2" --no-cache-dir -q
!pip install "accelerate==0.30.0" -q
!pip install sacrebleu sqlglot -q

In [3]:
import torch
import transformers, peft, bitsandbytes
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")
print(f"transformers: {transformers.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")

CUDA: True, GPU: Tesla T4
transformers: 4.43.4
bitsandbytes: 0.49.2


In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("✅ Logged in")

✅ Logged in


In [7]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

# Find checkpoint-175 path
ck175_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        if d == 'checkpoint-175':
            ck175_path = os.path.join(root, d)
            break

print(f"Checkpoint-175 path: {ck175_path}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B", trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, ck175_path)
model.eval()

print(f"✅ Loaded checkpoint-175")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Checkpoint-175 path: /kaggle/input/datasets/aniruddhabhagat/english-baseline-checkpoint-175/checkpoint-175


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

✅ Loaded checkpoint-175
GPU memory: 2.48 GB


In [8]:
import json, time, os

# Find hinglish test set
hinglish_test_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if 'hinglish' in full.lower() and f == 'test.json' and 'serialized' not in full:
            hinglish_test_path = full
            break

print(f"Hinglish test: {hinglish_test_path}")

with open(hinglish_test_path, encoding='utf-8') as f:
    hinglish_test = json.load(f)

print(f"Test set: {len(hinglish_test)} examples")

def generate_sql(prompt, max_new_tokens=150):
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_sql = tokenizer.decode(new_tokens, skip_special_tokens=True)
    for special in ['<|fim_middle|>','<|fim_prefix|>','<|fim_suffix|>',
                    '<|endoftext|>','<|im_end|>']:
        generated_sql = generated_sql.replace(special, '')
    return generated_sql.split('\n')[0].strip()

predictions_en_175 = []
start = time.time()

for i, row in enumerate(hinglish_test):
    pred = generate_sql(row['prompt'])
    predictions_en_175.append({
        'db_id': row['db_id'],
        'question_hinglish': row['question_hinglish'],
        'prediction': pred,
        'gold': row['completion']
    })
    if i % 50 == 0:
        print(f"Progress: {i+1}/{len(hinglish_test)} — {time.time()-start:.0f}s")

print(f"✅ Done — {len(predictions_en_175)} predictions")

os.makedirs('/kaggle/working/results', exist_ok=True)
with open('/kaggle/working/results/english_baseline_ck175_predictions.json',
          'w', encoding='utf-8') as f:
    json.dump(predictions_en_175, f, ensure_ascii=False, indent=2)
print("✅ Saved")

Hinglish test: /kaggle/input/datasets/aniruddhabhagat/hinglishdb-finetuning-data/hinglish_finetuning_ready/test.json
Test set: 381 examples


2026-07-01 02:00:07.144803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782871207.554998      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782871207.670594      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782871208.586742      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782871208.586774      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782871208.586777      58 computation_placer.cc:177] computation placer alr

Progress: 1/381 — 31s
Progress: 51/381 — 636s
Progress: 101/381 — 1215s
Progress: 151/381 — 1764s
Progress: 201/381 — 2451s
Progress: 251/381 — 3222s
Progress: 301/381 — 3963s
Progress: 351/381 — 4549s
✅ Done — 381 predictions
✅ Saved


In [9]:
from sacrebleu.metrics import BLEU
import re, sqlglot
from sqlglot import exp

def compute_bleu(predictions):
    bleu = BLEU(tokenize="none")
    return bleu.corpus_score(
        [p['prediction'] for p in predictions],
        [[p['gold'] for p in predictions]]
    ).score

def parse_sql_clauses(sql):
    sql = sql.strip().rstrip(';')
    patterns = {
        'SELECT':   r'SELECT\s+(.*?)\s+FROM',
        'FROM':     r'FROM\s+(.*?)(?=\s+WHERE|\s+GROUP BY|\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'WHERE':    r'WHERE\s+(.*?)(?=\s+GROUP BY|\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'GROUP BY': r'GROUP BY\s+(.*?)(?=\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'HAVING':   r'HAVING\s+(.*?)(?=\s+ORDER BY|\s+LIMIT|$)',
        'ORDER BY': r'ORDER BY\s+(.*?)(?=\s+LIMIT|$)',
    }
    clauses = {}
    for clause, pattern in patterns.items():
        match = re.search(pattern, sql, re.IGNORECASE | re.DOTALL)
        clauses[clause] = match.group(1).strip() if match else ''
    return clauses

def split_where_clause(text):
    if not text: return set()
    parts = re.split(r'\s+(?:AND|OR)\s+', text, flags=re.IGNORECASE)
    return set(re.sub(r'\bt\d+\.', '', p.strip().lower()) for p in parts)

def split_select_from_clause(text):
    if not text: return set()
    return set(
        re.sub(r'`', '', re.sub(r'\bt\d+\.', '', p.strip().lower()))
        for p in text.split(',')
    )

def compute_sqam(pred_sql, gold_sql):
    weights = {'SELECT':0.3,'FROM':0.2,'WHERE':0.25,
               'GROUP BY':0.1,'HAVING':0.05,'ORDER BY':0.1}
    pred_c = parse_sql_clauses(pred_sql)
    gold_c = parse_sql_clauses(gold_sql)
    score = 0.0
    for clause, weight in weights.items():
        if clause in ('WHERE','HAVING'):
            pred_set = split_where_clause(pred_c[clause])
            gold_set = split_where_clause(gold_c[clause])
        else:
            pred_set = split_select_from_clause(pred_c[clause])
            gold_set = split_select_from_clause(gold_c[clause])
        if not gold_set and not pred_set: clause_score = 1.0
        elif not gold_set or not pred_set: clause_score = 0.0
        else:
            i = len(pred_set & gold_set)
            u = len(pred_set | gold_set)
            clause_score = i/u if u > 0 else 0.0
        score += weight * clause_score
    return score

def ast_to_node_list(node):
    nodes = []
    def walk(n, depth=0):
        if n is None: return
        node_repr = type(n).__name__
        if hasattr(n, 'this') and isinstance(n.this, str):
            node_repr += f":{n.this.lower()}"
        nodes.append((depth, node_repr))
        for child in n.args.values():
            if isinstance(child, list):
                for c in child:
                    if isinstance(c, exp.Expression): walk(c, depth+1)
            elif isinstance(child, exp.Expression):
                walk(child, depth+1)
    walk(node)
    return nodes

def tree_edit_distance(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if a[i-1]==b[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[m][n]

def compute_tsed(pred_sql, gold_sql):
    try:
        pred_ast = sqlglot.parse_one(pred_sql, read="sqlite")
        gold_ast = sqlglot.parse_one(gold_sql, read="sqlite")
    except Exception:
        return 0.0
    pred_nodes = ast_to_node_list(pred_ast)
    gold_nodes = ast_to_node_list(gold_ast)
    D = tree_edit_distance(pred_nodes, gold_nodes)
    N = max(len(pred_nodes), len(gold_nodes))
    return max(0.0, 1.0 - D/N) if N > 0 else 1.0

# Score ck-175
bleu_175     = compute_bleu(predictions_en_175)
sqam_175     = sum(compute_sqam(p['prediction'], p['gold']) for p in predictions_en_175) / len(predictions_en_175)
tsed_175     = sum(compute_tsed(p['prediction'], p['gold']) for p in predictions_en_175) / len(predictions_en_175)

# Known results
bleu_ck25_en = 12.63;  sqam_ck25_en = 0.241;  tsed_ck25_en = 0.285
bleu_hi      = 20.39;  sqam_hi      = 0.319;  tsed_hi      = 0.308

best_en_bleu = max(bleu_ck25_en, bleu_175)
best_en_sqam = max(sqam_ck25_en, sqam_175)
best_en_tsed = max(tsed_ck25_en, tsed_175)

print("=" * 65)
print("EX2: English baseline checkpoints vs Hinglish fine-tuned")
print("=" * 65)
print(f"\n{'Model':<38} {'BLEU':>6} {'SQAM':>7} {'TSED':>7}")
print("-" * 62)
print(f"{'English baseline ck-25':<38} {bleu_ck25_en:>6.2f} {sqam_ck25_en:>7.4f} {tsed_ck25_en:>7.4f}")
print(f"{'English baseline ck-175':<38} {bleu_175:>6.2f} {sqam_175:>7.4f} {tsed_175:>7.4f}")
print(f"{'Hinglish fine-tuned ck-125':<38} {bleu_hi:>6.2f} {sqam_hi:>7.4f} {tsed_hi:>7.4f}")
print("-" * 62)
print(f"\nGap (Hinglish ck-125 vs best English checkpoint):")
print(f"  BLEU: {bleu_hi - best_en_bleu:+.2f}")
print(f"  SQAM: {sqam_hi - best_en_sqam:+.4f}")
print(f"  TSED: {tsed_hi - best_en_tsed:+.4f}")
print(f"\nDarijaDB reference gap (English - Darija): BLEU -10.1%")
print(f"IndicDB reference gap (English - Hinglish): -6.87% EX")

# Save
import json
results = {
    'experiment': 'EX2 — English baseline vs Hinglish fine-tuned',
    'english_ck25':   {'bleu': bleu_ck25_en, 'sqam': sqam_ck25_en, 'tsed': tsed_ck25_en},
    'english_ck175':  {'bleu': round(bleu_175,2), 'sqam': round(sqam_175,4),
                       'tsed': round(tsed_175,4)},
    'hinglish_ck125': {'bleu': bleu_hi, 'sqam': sqam_hi, 'tsed': tsed_hi},
    'gap_vs_best_english': {
        'bleu': round(bleu_hi - best_en_bleu, 2),
        'sqam': round(sqam_hi - best_en_sqam, 4),
        'tsed': round(tsed_hi - best_en_tsed, 4)
    }
}
with open('/kaggle/working/results/ex2_complete.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\n✅ Saved to /kaggle/working/results/ex2_complete.json")
print("Download from Kaggle Output panel and upload to Drive")

EX2: English baseline checkpoints vs Hinglish fine-tuned

Model                                    BLEU    SQAM    TSED
--------------------------------------------------------------
English baseline ck-25                  12.63  0.2410  0.2850
English baseline ck-175                 22.71  0.3257  0.4045
Hinglish fine-tuned ck-125              20.39  0.3190  0.3080
--------------------------------------------------------------

Gap (Hinglish ck-125 vs best English checkpoint):
  BLEU: -2.32
  SQAM: -0.0067
  TSED: -0.0965

DarijaDB reference gap (English - Darija): BLEU -10.1%
IndicDB reference gap (English - Hinglish): -6.87% EX

✅ Saved to /kaggle/working/results/ex2_complete.json
Download from Kaggle Output panel and upload to Drive
